# 04 Modelo Regresion Total Pedido

Este notebook entrena el segundo modelo del proyecto usando **Regresion Lineal** para estimar el `total_pedido`.

## 1. Librerias

Se importan las librerias necesarias para la regresion.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Cargar la base EDA

Se usa el parquet de la etapa 02 como entrada para el modelo de regresion.

In [ ]:
# Definir la ruta base del proyecto.
PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Apuntar al parquet del EDA.
PARQUET_PATH = PROJECT_ROOT / "parquets" / "02_EDA_Base_Tickets" / "02_base_eda_tickets.parquet"

# Cargar la base para regresion.
df = pd.read_parquet(PARQUET_PATH)
df.head()

## 3. Seleccionar variables y target

Aqui se definen las variables explicativas y el objetivo `total_pedido`.

In [ ]:
# Definir las variables explicativas del modelo.
FEATURES = [
    "dia", "mes", "trimestre", "dia_semana", "dia_tipo", "fin_semana",
    "ciudad", "capacidad_sucursal", "tipo_empleado", "salario", "turno",
    "numero_mesa", "capacidad_mesa", "metodo_pago", "lineas_ticket",
    "cantidad_total", "platillos_distintos", "categorias_distintas",
    "incluye_bebida", "incluye_postre", "incluye_entrada", "incluye_platillo_fuerte"
]

# Definir la variable objetivo.
TARGET = "total_pedido"

# Separar X y y para el entrenamiento.
X = df[FEATURES].copy()
y = df[TARGET].copy()

## 4. Particion de entrenamiento y prueba

Se divide la muestra para poder medir el error del modelo.

In [ ]:
# Dividir la base en entrenamiento y prueba.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Revisar dimensiones de las particiones.
X_train.shape, X_test.shape

## 5. Preprocesamiento y modelo

Se construye un pipeline con transformaciones y regresion lineal.

In [ ]:
# Separar columnas numericas y categoricas.
numericas = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categoricas = X.select_dtypes(include=["object"]).columns.tolist()

# Definir el preprocesador del pipeline.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numericas,
        ),
        (
            "cat",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categoricas,
        ),
    ]
)

# Definir el pipeline completo de regresion.
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])

# Entrenar el modelo.
pipeline.fit(X_train, y_train)

## 6. Evaluacion del modelo

Se calculan errores y el poder explicativo del modelo sobre la muestra de prueba.

In [ ]:
# Generar predicciones sobre test.
y_pred = pipeline.predict(X_test)

# Resumir metricas principales.
metricas = {
    "mae": round(float(mean_absolute_error(y_test, y_pred)), 4),
    "rmse": round(float(np.sqrt(mean_squared_error(y_test, y_pred))), 4),
    "r2": round(float(r2_score(y_test, y_pred)), 4),
}

pd.DataFrame(metricas.items(), columns=["metrica", "valor"])

## 7. Revisar algunas predicciones

Esta vista permite comparar el total real contra el estimado.

In [ ]:
# Construir una vista rapida de resultados sobre test.
resultados_test = X_test.copy()
resultados_test["total_real"] = y_test
resultados_test["total_pred"] = y_pred
resultados_test["residuo"] = resultados_test["total_real"] - resultados_test["total_pred"]

# Mostrar las primeras filas de resultados.
resultados_test.head()

## 6. Visualizaciones del modelo

Estas graficas ayudan a comparar el total real contra el estimado por el modelo.


In [ ]:
# Graficar comparacion entre total real y total estimado.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=resultados_test, x="total_real", y="total_pred", ax=axes[0])
min_val = min(resultados_test["total_real"].min(), resultados_test["total_pred"].min())
max_val = max(resultados_test["total_real"].max(), resultados_test["total_pred"].max())
axes[0].plot([min_val, max_val], [min_val, max_val], "r--")
axes[0].set_title("Total real vs total predicho")
axes[0].set_xlabel("total_real")
axes[0].set_ylabel("total_pred")

# Graficar distribucion del residuo.
sns.histplot(resultados_test["residuo"], bins=30, kde=True, ax=axes[1])
axes[1].set_title("Distribucion del residuo")
axes[1].set_xlabel("residuo")
plt.tight_layout()
